# Depth-Guided CC Instance Generation — Step-by-Step Visualization

Visualizes each step of `build_instance_map_depth_cc()` from `convert_to_cups_format.py`.

**Algorithm**: Split thing-class clusters at DepthPro depth edges, then reclaim all pixels via dilation + nearest-neighbor. Produces 65% more instances than plain CC while maintaining 100% thing-pixel coverage.

**4 Cityscapes val images**: frankfurt_000000_000294, frankfurt_000000_001236, lindau_000000_000019, munster_000000_000019

In [ ]:
import numpy as np
from PIL import Image
from scipy import ndimage
from scipy.ndimage import gaussian_filter, sobel, distance_transform_edt
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

In [ ]:
# ─── Paths ───
CS_ROOT = Path('/Users/qbit-glitch/Desktop/datasets/cityscapes')
DEPTH_DIR = CS_ROOT / 'depth_depthpro' / 'val'
SEM_DIR = CS_ROOT / 'pseudo_semantic_raw_k80' / 'val'
IMG_DIR = CS_ROOT / 'leftImg8bit' / 'val'
GT_DIR = CS_ROOT / 'gtFine' / 'val'
CENTROIDS_PATH = CS_ROOT / 'pseudo_semantic_raw_k80' / 'kmeans_centroids.npz'

# ─── 4 images to visualize ───
IMAGES = [
    ('frankfurt', 'frankfurt_000000_000294'),
    ('frankfurt', 'frankfurt_000000_001236'),
    ('lindau',    'lindau_000000_000019'),
    ('munster',   'munster_000000_000019'),
]

# ─── DepthPro optimal parameters ───
TAU = 0.01           # Sobel gradient threshold
SIGMA = 0.0          # No blur for DepthPro
DILATION_ITERS = 3   # Boundary reclamation
MIN_AREA = 50        # Low threshold for full coverage

# ─── Load centroids → thing cluster IDs ───
data = np.load(CENTROIDS_PATH)
cluster_to_class = data['cluster_to_class']
THING_TRAINIDS = {11, 12, 13, 14, 15, 16, 17, 18}
THING_CLUSTER_IDS = {i for i, c in enumerate(cluster_to_class) if int(c) in THING_TRAINIDS}

print(f'Thing cluster IDs ({len(THING_CLUSTER_IDS)}/80): {sorted(THING_CLUSTER_IDS)}')
print(f'Mapped to trainIDs: {sorted(set(int(cluster_to_class[i]) for i in THING_CLUSTER_IDS))}')

In [ ]:
# ─── Helper Functions ───

def random_colormap(n_ids, seed=42):
    """Generate random colors for instance IDs. ID 0 = black (background)."""
    rng = np.random.RandomState(seed)
    colors = np.zeros((n_ids + 1, 3))
    for i in range(1, n_ids + 1):
        colors[i] = rng.rand(3) * 0.8 + 0.2  # avoid too dark
    return colors


def colorize_instances(instance_map, max_id=None):
    """Convert instance map to RGB image with random colors per ID."""
    if max_id is None:
        max_id = int(instance_map.max())
    colors = random_colormap(max_id)
    h, w = instance_map.shape
    rgb = np.zeros((h, w, 3))
    for i in range(max_id + 1):
        mask = instance_map == i
        if mask.any():
            rgb[mask] = colors[min(i, len(colors)-1)]
    return rgb


def colorize_thing_clusters(semantic, thing_ids, cluster_to_class):
    """Color thing clusters by their trainID, stuff as gray."""
    # Cityscapes thing colors (trainID 11-18)
    thing_colors = {
        11: [220, 20, 60],    # person - crimson
        12: [255, 0, 0],      # rider - red
        13: [0, 0, 142],      # car - dark blue
        14: [0, 0, 70],       # truck - navy
        15: [0, 60, 100],     # bus - dark cyan
        16: [0, 80, 100],     # train - teal
        17: [0, 0, 230],      # motorcycle - blue
        18: [119, 11, 32],    # bicycle - dark red
    }
    h, w = semantic.shape
    rgb = np.full((h, w, 3), 40, dtype=np.uint8)  # gray background
    for cid in thing_ids:
        mask = semantic == cid
        if mask.any():
            tid = int(cluster_to_class[cid])
            color = thing_colors.get(tid, [200, 200, 200])
            rgb[mask] = color
    return rgb


def build_cc_instances(semantic, thing_ids, min_area=50):
    """Plain CC instances (no depth). For comparison."""
    H, W = semantic.shape
    instance_map = np.zeros((H, W), dtype=np.uint16)
    iid = 1
    for cid in sorted(thing_ids):
        mask = semantic == cid
        if mask.sum() == 0:
            continue
        labeled, n = ndimage.label(mask)
        for c in range(1, n + 1):
            cm = labeled == c
            if cm.sum() >= min_area:
                instance_map[cm] = iid
                iid += 1
    return instance_map


print('Helpers loaded.')

In [ ]:
def visualize_depth_guided_cc(city, stem, figsize=(28, 16)):
    """Full step-by-step visualization for one image."""
    
    # ─── Load data ───
    rgb = np.array(Image.open(IMG_DIR / city / f'{stem}_leftImg8bit.png'))
    depth = np.load(DEPTH_DIR / city / f'{stem}.npy')
    semantic = np.array(Image.open(SEM_DIR / city / f'{stem}.png'))
    
    # Resize to depth resolution (512x1024)
    H, W = depth.shape
    rgb_resized = np.array(Image.fromarray(rgb).resize((W, H), Image.BILINEAR))
    if semantic.shape != (H, W):
        semantic = np.array(Image.fromarray(semantic).resize((W, H), Image.NEAREST))
    
    # ═══ STEP 1: Compute depth edges ═══
    depth_smooth = depth.astype(np.float64)  # sigma=0.0, no blur
    gx = sobel(depth_smooth, axis=1)
    gy = sobel(depth_smooth, axis=0)
    grad_mag = np.sqrt(gx**2 + gy**2)
    depth_edges = grad_mag > TAU
    
    # ═══ STEP 2: Thing cluster mask ═══
    thing_mask = np.zeros((H, W), dtype=bool)
    for cid in THING_CLUSTER_IDS:
        thing_mask |= (semantic == cid)
    
    # ═══ STEP 3: Split mask (remove depth edges from things) ═══
    split_mask = thing_mask & ~depth_edges
    
    # ═══ STEP 4: Depth-guided CC instances ═══
    instance_map_depth = np.zeros((H, W), dtype=np.uint16)
    iid = 1
    for cid in sorted(THING_CLUSTER_IDS):
        cls_mask = semantic == cid
        if cls_mask.sum() == 0:
            continue
        s_mask = cls_mask & ~depth_edges
        labeled, n = ndimage.label(s_mask)
        cc_list = []
        for c in range(1, n + 1):
            cm = labeled == c
            area = int(cm.sum())
            if area >= MIN_AREA:
                cc_list.append((cm, area))
        cc_list.sort(key=lambda x: -x[1])
        
        assigned = np.zeros((H, W), dtype=bool)
        for cm, area in cc_list:
            dilated = ndimage.binary_dilation(cm, iterations=DILATION_ITERS)
            reclaimed = dilated & cls_mask & ~assigned
            final = cm | reclaimed
            if final.sum() >= MIN_AREA:
                instance_map_depth[final] = iid
                assigned |= final
                iid += 1
        
        # Reclaim remaining via nearest-neighbor
        unassigned = cls_mask & ~assigned
        if unassigned.sum() > 0 and assigned.sum() > 0:
            _, nearest_idx = distance_transform_edt(~assigned, return_distances=True,
                                                     return_indices=True)
            for y, x in zip(*np.where(unassigned)):
                ny, nx = nearest_idx[0, y, x], nearest_idx[1, y, x]
                if instance_map_depth[ny, nx] > 0:
                    instance_map_depth[y, x] = instance_map_depth[ny, nx]
    
    # ═══ STEP 5: Plain CC instances (for comparison) ═══
    instance_map_cc = build_cc_instances(semantic, THING_CLUSTER_IDS, MIN_AREA)
    
    # ─── Stats ───
    n_depth = int(instance_map_depth.max())
    n_cc = int(instance_map_cc.max())
    cov_depth = (instance_map_depth > 0).sum()
    cov_cc = (instance_map_cc > 0).sum()
    thing_px = thing_mask.sum()
    
    # ═══ PLOT ═══
    fig, axes = plt.subplots(2, 4, figsize=figsize)
    fig.suptitle(f'{stem}\n'
                 f'Depth-guided CC: {n_depth} instances | Plain CC: {n_cc} instances | '
                 f'Thing pixels: {thing_px:,}',
                 fontsize=14, fontweight='bold')
    
    # Row 1: Algorithm steps
    ax = axes[0, 0]
    ax.imshow(rgb_resized)
    ax.set_title('Step 0: Original Image', fontsize=11)
    ax.axis('off')
    
    ax = axes[0, 1]
    ax.imshow(depth, cmap='viridis')
    ax.set_title('Step 1: DepthPro Depth Map', fontsize=11)
    ax.axis('off')
    
    ax = axes[0, 2]
    ax.imshow(grad_mag, cmap='hot', vmin=0, vmax=0.05)
    ax.set_title(f'Step 2: Sobel Gradient (\u03c4={TAU})', fontsize=11)
    ax.axis('off')
    
    ax = axes[0, 3]
    edge_vis = np.zeros((H, W, 3), dtype=np.uint8)
    edge_vis[depth_edges] = [255, 255, 255]
    # Overlay edges on dimmed RGB
    overlay = (rgb_resized * 0.3).astype(np.uint8)
    overlay[depth_edges] = [255, 100, 50]
    ax.imshow(overlay)
    ax.set_title(f'Step 3: Depth Edges (\u03c4={TAU})', fontsize=11)
    ax.axis('off')
    
    # Row 2: Thing masks and instances
    ax = axes[1, 0]
    thing_vis = colorize_thing_clusters(semantic, THING_CLUSTER_IDS, cluster_to_class)
    ax.imshow(thing_vis)
    ax.set_title(f'Step 4: Thing Clusters ({thing_px:,} px)', fontsize=11)
    ax.axis('off')
    
    ax = axes[1, 1]
    split_vis = np.full((H, W, 3), 20, dtype=np.uint8)
    split_vis[split_mask] = [0, 200, 100]
    split_vis[thing_mask & depth_edges] = [255, 50, 50]  # edges in red
    ax.imshow(split_vis)
    ax.set_title('Step 5: Split Mask (green) + Edges (red)', fontsize=11)
    ax.axis('off')
    
    ax = axes[1, 2]
    max_id = max(n_depth, n_cc)
    ax.imshow(colorize_instances(instance_map_cc, max_id))
    ax.set_title(f'Plain CC: {n_cc} instances (no depth)', fontsize=11)
    ax.axis('off')
    
    ax = axes[1, 3]
    ax.imshow(colorize_instances(instance_map_depth, max_id))
    ax.set_title(f'Depth-Guided CC: {n_depth} instances (+{n_depth-n_cc})', fontsize=11,
                 color='darkgreen' if n_depth > n_cc else 'black')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'stem': stem,
        'n_depth': n_depth,
        'n_cc': n_cc,
        'thing_px': thing_px,
        'cov_depth': cov_depth,
        'cov_cc': cov_cc,
    }

print('Visualization function ready.')

## Image 1: frankfurt_000000_000294
Urban scene with cars and pedestrians

In [ ]:
stats1 = visualize_depth_guided_cc('frankfurt', 'frankfurt_000000_000294')

## Image 2: frankfurt_000000_001236
Street with multiple vehicles

In [ ]:
stats2 = visualize_depth_guided_cc('frankfurt', 'frankfurt_000000_001236')

## Image 3: lindau_000000_000019
Small town, fewer objects

In [ ]:
stats3 = visualize_depth_guided_cc('lindau', 'lindau_000000_000019')

## Image 4: munster_000000_000019
City center with bikes and people

In [ ]:
stats4 = visualize_depth_guided_cc('munster', 'munster_000000_000019')

## Summary Statistics

In [ ]:
all_stats = [stats1, stats2, stats3, stats4]

print(f'{"Image":<35s} {"Plain CC":>10s} {"Depth CC":>10s} {"Extra":>7s} {"Thing px":>10s} {"CC cov%":>8s} {"Depth cov%":>10s}')
print('-' * 95)
for s in all_stats:
    cc_pct = s['cov_cc'] / max(s['thing_px'], 1) * 100
    dp_pct = s['cov_depth'] / max(s['thing_px'], 1) * 100
    print(f"{s['stem']:<35s} {s['n_cc']:>10d} {s['n_depth']:>10d} {s['n_depth']-s['n_cc']:>+7d} {s['thing_px']:>10,d} {cc_pct:>7.1f}% {dp_pct:>9.1f}%")

print()
avg_cc = np.mean([s['n_cc'] for s in all_stats])
avg_dp = np.mean([s['n_depth'] for s in all_stats])
print(f'Average: Plain CC = {avg_cc:.1f} instances, Depth CC = {avg_dp:.1f} instances ({(avg_dp/avg_cc-1)*100:+.1f}%)')